# PageRank from Scratch
Power iteration on a small web graph. (PageRank wasn't in your source files, so this notebook is written from scratch.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Graph
Edges are `(from, to)` links between 4 pages.

In [ ]:
edges = [(0, 1), (0, 2), (1, 2), (2, 0), (3, 2)]
n = 4
A = np.zeros((n, n))
for i, j in edges:
    A[i, j] = 1
print(A)

### Transition matrix
Each row is split evenly over a page's outgoing links. A *dangling* page (no outlinks) jumps to any page with equal probability.

In [ ]:
out = A.sum(axis=1, keepdims=True)
P = np.where(out == 0, 1 / n, A / np.maximum(out, 1))
print(P)

### Power iteration
$r \leftarrow d\,(rP) + \frac{1-d}{n}$, where $d = 0.85$ is the damping factor (85% follow links, 15% teleport randomly).

In [ ]:
def pagerank(P, d=0.85, tol=1e-10, max_iter=1000):
    n = len(P)
    r = np.full(n, 1 / n)
    history = [r]
    for _ in range(max_iter):
        r_new = d * (r @ P) + (1 - d) / n
        history.append(r_new)
        if np.abs(r_new - r).sum() < tol:
            break
        r = r_new
    return r_new, np.array(history)

ranks, history = pagerank(P)
for i, s in enumerate(ranks):
    print(f"Node {i}: {s:.6f}")
print("Sum of scores:", ranks.sum().round(6), "| Iterations:", len(history) - 1)

In [ ]:
plt.plot(history, label=[f"Node {i}" for i in range(n)])
plt.xlabel("Iteration"); plt.ylabel("PageRank score"); plt.title("Convergence"); plt.legend(); plt.show()

### Interpretation
Node 2 is linked to by every other page, so it gets the highest score. Node 3 has no incoming links, so it only receives the teleportation share $(1-d)/n$, the lowest score possible.